# SENTINEL v3 — Contrastive Augmentation Training
**Run cells in order. Set DATASET_NAME before running.**

In [ ]:
# ── Cell 1: Setup ──
from google.colab import drive
drive.mount('/content/drive')

import sys, torch
GITHUB_TOKEN = 'your_token_here'  # paste your token
!git clone https://{GITHUB_TOKEN}@github.com/dhanya-iiitk/SENTINEL-LogAD.git /content/SENTINEL
sys.path.insert(0, '/content/SENTINEL')

!pip install drain3 groq scikit-learn torch-scatter torch-sparse torch-geometric \
    -f https://data.pyg.org/whl/torch-{torch.__version__.split('+')[0]}+cu{torch.version.cuda.replace('.','')}.html -q

import random
import numpy as np
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'GPU    : {torch.cuda.get_device_name(0)}')
print('Setup done ✓')

In [ ]:
# ── Cell 2: Config ──
from configs.config import cfg
import os

DATASET_NAME = 'HDFS'    # Change to 'BGL' or 'OpenStack'
DEV_MODE     = True      # True = 2000 samples, False = full

PARSED_PATHS = {
    'HDFS':      f'{cfg.PARSED_DIR}/hdfs_parsed.pkl',
    'BGL':       f'{cfg.PARSED_DIR}/bgl_parsed.pkl',
    'OpenStack': f'{cfg.PARSED_DIR}/openstack_parsed.pkl',
}
VOCAB_PATHS = {
    'HDFS':      f'{cfg.PARSED_DIR}/hdfs_parser.pkl',
    'BGL':       f'{cfg.PARSED_DIR}/bgl_parser.pkl',
    'OpenStack': f'{cfg.PARSED_DIR}/openstack_parser.pkl',
}

# Check parsed files exist
for name, path in PARSED_PATHS.items():
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1e6 if exists else 0
    print(f"{'✅' if exists else '❌'} {name}: {size:.1f} MB")

In [ ]:
# ── Cell 3: Load Data + Build DataLoaders ──
import pickle
from sentinel.data.parser import load_parsed
from sentinel.data.contrastive_dataset import (
    split_sessions, get_dataloaders_v3
)

# Load parsed sessions
obj      = load_parsed(PARSED_PATHS[DATASET_NAME])
sessions = obj['sessions']
labels   = obj['labels']
print(f'Loaded {len(sessions):,} sessions')

# Get vocab size
with open(VOCAB_PATHS[DATASET_NAME], 'rb') as f:
    vocab_obj = pickle.load(f)
VOCAB_SIZE = vocab_obj['next_id'] + 10
print(f'Vocab size: {VOCAB_SIZE}')

# Split
train_data, val_data, test_data = split_sessions(
    sessions, labels,
    dev_mode    = DEV_MODE,
    dev_samples = cfg.DEV_SAMPLES,
    seed        = cfg.SEED,
)

# DataLoaders
train_loader, val_loader, test_loader = get_dataloaders_v3(
    train_data, val_data, test_data,
    vocab_size  = VOCAB_SIZE,
    max_seq_len = cfg.MAX_SEQ_LEN,
    batch_size  = cfg.BATCH_SIZE,
)

In [ ]:
# ── Cell 4: Verify Augmentation ──
from sentinel.data.contrastive_dataset import augment_session

sample = sessions[0][:15]
print(f'Original  : {sample}')
print(f'Shuffled  : {augment_session(sample, VOCAB_SIZE, "shuffle")}')
print(f'Replaced  : {augment_session(sample, VOCAB_SIZE, "replace")}')
print(f'Inserted  : {augment_session(sample, VOCAB_SIZE, "insert")}')

# Check dataloader batch
batch = next(iter(train_loader))
print(f'\nBatch shapes:')
print(f'  seq_norm : {batch["seq_norm"].shape}')
print(f'  seq_aug  : {batch["seq_aug"].shape}')
print(f'  graph_norm nodes: {batch["graph_norm"].x.shape}')
print(f'  graph_aug  nodes: {batch["graph_aug"].x.shape}')
print('Augmentation OK ✓')

In [ ]:
# ── Cell 5: Build Model ──
from sentinel.models.sentinel_v3 import SENTINELv3

model = SENTINELv3(
    vocab_size         = VOCAB_SIZE,
    embed_dim          = cfg.EMBED_DIM,
    pad_idx            = cfg.PAD_IDX,
    transformer_layers = cfg.TRANSFORMER_LAYERS,
    num_heads          = cfg.ATTENTION_HEADS,
    ffn_dim            = cfg.FFN_DIM,
    gnn_hidden         = cfg.GNN_HIDDEN,
    gnn_layers         = cfg.GNN_LAYERS,
    latent_dim         = cfg.FUSION_OUTPUT,
    flow_layers        = 6,
    flow_hidden        = 512,
    dropout            = cfg.DROPOUT,
    margin             = 10.0,
    lambda_contrast    = 0.5,
    lambda_recon       = 0.1,
).to(DEVICE)

print(f'Parameters: {model.count_parameters():,}')

# Forward pass test
batch_dev = {
    'seq_norm':   batch['seq_norm'].to(DEVICE),
    'seq_aug':    batch['seq_aug'].to(DEVICE),
    'len_norm':   batch['len_norm'].to(DEVICE),
    'len_aug':    batch['len_aug'].to(DEVICE),
    'graph_norm': batch['graph_norm'].to(DEVICE),
    'graph_aug':  batch['graph_aug'].to(DEVICE),
}
total, fl, cl, rl = model(batch_dev)
print(f'Forward pass OK ✓')
print(f'  Total loss    : {total.item():.4f}')
print(f'  Flow loss     : {fl.item():.4f}')
print(f'  Contrast loss : {cl.item():.4f}')
print(f'  Recon loss    : {rl.item():.4f}')

In [ ]:
# ── Cell 6: Train ──
from sentinel.utils.trainer_v3 import train_sentinel_v3

model, history = train_sentinel_v3(
    model, train_loader, val_loader,
    epochs         = cfg.EPOCHS,
    lr             = cfg.LEARNING_RATE,
    weight_decay   = cfg.WEIGHT_DECAY,
    patience       = 15,
    use_amp        = cfg.USE_AMP,
    device         = DEVICE,
    checkpoint_dir = f'{cfg.RESULTS_DIR}/checkpoints',
    dataset_name   = DATASET_NAME,
)

In [ ]:
# ── Cell 7: Evaluate ──
import json
from sentinel.utils.trainer_v3 import (
    collect_scores, compute_youden_threshold,
    compute_metrics, print_metrics,
    plot_training_v3, plot_score_dist
)

# Training curves
plot_training_v3(
    history,
    save_path    = f'{cfg.RESULTS_DIR}/{DATASET_NAME}_v3_train.png',
    dataset_name = DATASET_NAME,
)

# YOBDM threshold
val_scores, val_labels   = collect_scores(model, val_loader, DEVICE)
threshold                = compute_youden_threshold(val_scores, val_labels)

# Test evaluation
test_scores, test_labels = collect_scores(model, test_loader, DEVICE)
metrics                  = compute_metrics(test_scores, test_labels, threshold)
print_metrics(metrics, split=f'{DATASET_NAME} v3')

# Score distribution
plot_score_dist(
    test_scores, test_labels, threshold,
    save_path    = f'{cfg.RESULTS_DIR}/{DATASET_NAME}_v3_scores.png',
    dataset_name = DATASET_NAME,
)

# Save results
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f'{cfg.RESULTS_DIR}/{DATASET_NAME}_v3_results.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Results saved ✓')